In [1]:
import numpy as np
import plotly.express as px
import math
import pandas as pd


In [2]:

def get_atan_LUT(iterations=16):
    return np.array([math.atan(2 ** -i) for i in range(iterations)])

def get_fi_LUT(iterations=16):
    return np.round(get_atan_LUT(iterations)*32768) # Q1.15 beccause the thing needs negative nubers later

def cordic_prerotate(x, y):
    """
    Pre-rotates vector (x, y) into the right half-plane (x >= 0)
    to allow standard CORDIC loop convergence across all 4 quadrants.
    
    Returns: (x_new, y_new, z_offset)
    """
    if x < 0:
        if y >= 0:
            # Quadrant II: Rotate by -90° -> z offset = +90° (+pi/2)
            return y, -x, math.pi / 2.0
        else:
            # Quadrant III: Rotate by +90° -> z offset = -90° (-pi/2)
            return -y, x, -math.pi / 2.0
    else:
        # Quadrants I & IV: No rotation needed
        return x, y, 0.0
    
def cordic_atan(y, x=1.0, iterations=20):
    """
    Computes arctan(y/x) using the CORDIC algorithm in Vectoring Mode.
    
    Default x=1.0 allows computing direct atan(y).
    """
    # Precomputed look-up table for angles where tan(theta) = 2^(-i)
    atan_table = get_atan_LUT(iterations)
    x_curr, y_curr, z = cordic_prerotate(x, y)
    
    
    for i in range(iterations):
        # Determine rotation direction to drive y toward zero
        if y_curr < 0:
            d = 1   # Rotate counter-clockwise (positive angle)
        else:
            d = -1  # Rotate clockwise (negative angle)
        
        # Bit shifts: multiplying by 2^(-i)
        x_shift = x_curr * (2 ** -i)
        y_shift = y_curr * (2 ** -i)
        
        # Perform shift-and-add rotation
        x_curr = x_curr - d * y_shift
        y_curr = y_curr + d * x_shift
        
        # Accumulate the rotated angle
        z = z - d * atan_table[i]
        
    return z

In [3]:
test_pairs = [(0.5, 1.0), (1.0, 1.0), (3.0, 4.0), (-0.75, 1.0)]

print(f"{'Input (y, x)':>15} | {'CORDIC (deg)':>15} | {'math.atan (deg)':>18} | {'Error (deg)':>12}")
print("-" * 68)

for y, x in test_pairs:
    cordic_rad = cordic_atan(y, x)      # Computation stays in radians
    actual_rad = math.atan2(y, x)       # Computation stays in radians
    
    # Convert ONLY for the print output
    cordic_deg = math.degrees(cordic_rad)
    actual_deg = math.degrees(actual_rad)
    error_deg = abs(cordic_deg - actual_deg)
    
    print(f"({y:4.2f}, {x:4.2f})".rjust(15) + f" | {cordic_deg:13.4f}° | {actual_deg:16.4f}° | {error_deg:12.2e}")

   Input (y, x) |    CORDIC (deg) |    math.atan (deg) |  Error (deg)
--------------------------------------------------------------------
   (0.50, 1.00) |       26.5652° |          26.5651° |     1.07e-04
   (1.00, 1.00) |       44.9999° |          45.0000° |     9.38e-05
   (3.00, 4.00) |       36.8699° |          36.8699° |     2.46e-05
  (-0.75, 1.00) |      -36.8699° |         -36.8699° |     2.46e-05


In [4]:
n_points = 1000
t = np.linspace(0,20*math.pi,n_points)
angle = np.mod(np.linspace(0,8*math.pi,n_points), 2*math.pi)
sin_t=np.sin(angle)
cos_t=np.cos(angle)
df = pd.DataFrame({"angle":angle, "sin": sin_t, "cos": cos_t}, index=t)
# Apply row-by-row using a loop / lambda
df['cordic_angle'] = df.apply(lambda row: cordic_atan(row['sin'], row['cos'], 8), axis=1)

# Map output from [-pi, +pi] to [0, 2*pi) to match your input angle range
df['cordic_angle'] = np.mod(df['cordic_angle'], 2 * math.pi)

In [5]:
px.line(df, y=["sin", "cos", "cordic_angle"])

In [6]:
px.line(df, y=["angle", "cordic_angle"])